

# TMDB Enrichment for MovieLens

This notebook enriches MovieLens movies with selected TMDB metadata. It saves checkpoint CSV files during the run so progress is not lost if Colab disconnects.

In [ ]:
# Mount Google Drive so the MovieLens files and checkpoint files are available.
#from google.colab import drive
# drive.mount('/content/gdrive')

# Project data folder in Google Drive.
google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"
google_drive_path = "G:/My Drive/Colab Notebooks/Data/DSC_511_PROJECT/"  # for local testing

In [ ]:
# Core libraries for data handling, API requests, parallel execution, and checkpoints.
import os
import csv
import time
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

In [ ]:
# TMDB API key.
# Keep the key in one place so it is easy to replace or hide before submission.
TMDB_API_KEY = "55d206bac9cdb44a8069f01785a51884"

# Performance settings.
# SAMPLE_START / SAMPLE_SIZE control which batch to run in a single Colab session.
# Set SAMPLE_START=0 and SAMPLE_SIZE=None to process all movies in one go
# (may hit the Colab runtime limit for very large datasets).
# Alternatively, run batches of 10 000:
#   batch 1 → SAMPLE_START=0,     SAMPLE_SIZE=10000
#   batch 2 → SAMPLE_START=10000, SAMPLE_SIZE=10000  … and so on.
SAMPLE_START = 0
SAMPLE_SIZE = None  #10000  # set to None to process ALL movies at once
MAX_WORKERS = 20
SAVE_EVERY = 250

# Output files saved in Google Drive.
OUTPUT_DIR = Path(google_drive_path) / "enrichment_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = OUTPUT_DIR / "tmdb_metadata_selected.csv"
ENRICHED_MOVIES_FILE = OUTPUT_DIR / "movies_enriched_selected.csv"

print("Checkpoint file:", CHECKPOINT_FILE)
print("Enriched output:", ENRICHED_MOVIES_FILE)


Checkpoint file: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\tmdb_metadata_selected.csv
Enriched output: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\movies_enriched_selected.csv


In [ ]:
# Load MovieLens files.
# links.csv connects MovieLens movieId to TMDB and IMDb identifiers.
ratings_df = pd.read_csv(Path(google_drive_path) / "ratings.csv", usecols=["movieId"])
movies_df = pd.read_csv(Path(google_drive_path) / "movies.csv")
links_df = pd.read_csv(Path(google_drive_path) / "links.csv")

print("ratings rows:", len(ratings_df))
print("movies rows:", len(movies_df))
print("links rows:", len(links_df))

ratings rows: 33832162
movies rows: 86537
links rows: 86537


In [ ]:
# Build the list of movies to enrich from movies_df (all movies, not just rated ones).
# links_df provides the tmdbId mapping; movies without a valid tmdbId are kept
# in the final CSV but enriched columns will be empty (NaN).
links_with_tmdb = links_df[
    links_df["tmdbId"].notna()
].copy()

links_with_tmdb["movieId"] = links_with_tmdb["movieId"].astype(int)
links_with_tmdb["tmdbId"]  = links_with_tmdb["tmdbId"].astype(int)

# Merge movies_df with links to get the tmdbId per movie.
# We keep ALL movies (how='left') so movies without a tmdbId (e.g. movieId=126)
# are still present in the scrape list — they simply have no tmdbId and will be
# skipped by the API fetch, then retained as empty rows in the final CSV.
movies_with_links = movies_df.merge(links_with_tmdb[["movieId", "tmdbId", "imdbId"]],
                                    on="movieId", how="left")

# Restrict to movies that have a valid tmdbId for the API scrape.
links_filtered = movies_with_links[movies_with_links["tmdbId"].notna()].copy()
links_filtered["tmdbId"] = links_filtered["tmdbId"].astype(int)

# Apply optional batch window (change SAMPLE_START/SAMPLE_SIZE above).
if SAMPLE_SIZE is not None:
    links_filtered = links_filtered.iloc[SAMPLE_START:SAMPLE_START + SAMPLE_SIZE]

print("Total movies in movies.csv:", len(movies_df))
print("Movies with a valid tmdbId:", len(movies_with_links[movies_with_links['tmdbId'].notna()]))
print("Movies selected for this TMDB enrichment run:", len(links_filtered))
links_filtered.head()


Total movies in movies.csv: 86537
Movies with a valid tmdbId: 86411
Movies selected for this TMDB enrichment run: 86411


,movieId,title,genres,tmdbId,imdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862,114709.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844,113497.0
2,3,Grumpier Old Men (1995),Comedy|Romance,15602,113228.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357,114885.0
4,5,Father of the Bride Part II (1995),Comedy,11862,113041.0


In [ ]:
# Selected output fields.
# Only the columns needed for the project are fetched from TMDB.
FIELDNAMES = [
    "movieId",
    "tmdb_id",
    "budget",
    "revenue",
    "runtime",
    "release_date",
    "original_language",
    "director",
    "top_cast",
    "production_companies",
    "production_countries",
    "keywords"
]


In [ ]:
import pandas as pd
import csv
from pathlib import Path

# Assuming FIELDNAMES and CHECKPOINT_FILE are defined in previous cells or globally accessible
# Load previous checkpoint results if the file already exists.
# Rerunning the notebook continues from the missing movies only.
def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame(columns=FIELDNAMES), set()

    saved_df = pd.read_csv(path, low_memory=False)
    # Use pd.to_numeric with errors='coerce' to handle non-numeric values gracefully
    # Then drop NaNs and convert to integer type
    completed_ids = set(pd.to_numeric(saved_df["tmdb_id"], errors='coerce').dropna().astype(int))
    return saved_df, completed_ids


# Append rows to the checkpoint file without rewriting the whole file every time.
def append_checkpoint(path, rows):
    if not rows:
        return

    write_header = not path.exists() or path.stat().st_size == 0
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(rows)

saved_df, completed_tmdb_ids = load_checkpoint(CHECKPOINT_FILE)
print("Previously saved movies:", len(completed_tmdb_ids))

Previously saved movies: 4500


In [ ]:
# One HTTP session per worker thread improves speed by reusing connections.
thread_local = threading.local()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


def get_session():
    if not hasattr(thread_local, "session"):
        session = requests.Session()
        session.headers.update(HEADERS)
        thread_local.session = session
    return thread_local.session

In [ ]:
# Fetch selected TMDB metadata for one movie.
# append_to_response=credits,keywords brings director, cast, and keywords in the same request.
def fetch_tmdb_metadata(row, retries=3):
    movie_id = int(row["movieId"])
    tmdb_id  = int(row["tmdbId"])

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "append_to_response": "credits,keywords",
    }

    for attempt in range(retries):
        try:
            response = get_session().get(url, params=params, timeout=20)

            # Missing TMDB pages are skipped gracefully.
            if response.status_code == 404:
                return None

            # Rate limits are retried with a short wait.
            if response.status_code == 429:
                time.sleep(2 + attempt * 3)
                continue

            response.raise_for_status()
            data = response.json()

            crew = data.get("credits", {}).get("crew", [])
            cast = data.get("credits", {}).get("cast", [])

            directors = [p.get("name") for p in crew if p.get("job") == "Director"]

            release_date = data.get("release_date") or None

            # budget=0 is treated as missing by TMDB; store as None.
            raw_budget = data.get("budget")
            budget = raw_budget if raw_budget else None

            return {
                "movieId":           movie_id,
                "tmdb_id":           tmdb_id,
                "budget":            budget,
                "revenue":           data.get("revenue") or None,
                "runtime":           data.get("runtime") or None,
                "release_date":      release_date,
                "original_language": data.get("original_language"),
                "director":          "|".join(directors[:3]) if directors else None,
                # top 3 cast members
                "top_cast":          "|".join(p.get("name", "") for p in cast[:3]) if cast else None,
                "production_companies": "|".join(c.get("name", "") for c in data.get("production_companies", [])) or None,
                "production_countries": "|".join(c.get("name", "") for c in data.get("production_countries", [])) or None,
                "keywords":          "|".join(k.get("name", "") for k in data.get("keywords", {}).get("keywords", [])) or None,

            }

        except requests.RequestException:
            time.sleep(1 + attempt * 2)

    return None


In [ ]:
# Build the pending list by removing movies already saved in the checkpoint.
pending_df = links_filtered[~links_filtered["tmdbId"].isin(completed_tmdb_ids)].copy()

print("Movies in current run:", len(links_filtered))
print("Already completed:", len(completed_tmdb_ids))
print("Pending movies:", len(pending_df))

Movies in current run: 86411
Already completed: 4500
Pending movies: 81907


In [ ]:
pip install aiohttp


   --------------- ------------------------ 3/8 [attrs]
   ------------------------- -------------- 5/8 [yarl]
   ----------------------------------- ---- 7/8 [aiohttp]
   ----------------------------------- ---- 7/8 [aiohttp]
   ----------------------------------- ---- 7/8 [aiohttp]
   ---------------------------------------- 8/8 [aiohttp]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import aiohttp
import asyncio

SEMAPHORE_LIMIT = 30  # max concurrent requests

async def fetch_tmdb_async(session, row, semaphore, retries=3):
    movie_id = int(row["movieId"])
    tmdb_id = int(row["tmdbId"])
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY, "append_to_response": "credits,keywords"}

    for attempt in range(retries):
        try:
            async with semaphore:
                async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=20)) as resp:
                    if resp.status == 404:
                        return None
                    if resp.status == 429:
                        await asyncio.sleep(2 + attempt * 3)
                        continue
                    resp.raise_for_status()
                    data = await resp.json()

            crew = data.get("credits", {}).get("crew", [])
            cast = data.get("credits", {}).get("cast", [])
            directors = [p.get("name") for p in crew if p.get("job") == "Director"]
            raw_budget = data.get("budget")

            return {
                "movieId": movie_id,
                "tmdb_id": tmdb_id,
                "budget": raw_budget if raw_budget else None,
                "revenue": data.get("revenue") or None,
                "runtime": data.get("runtime") or None,
                "release_date": data.get("release_date") or None,
                "original_language": data.get("original_language"),
                "director": "|".join(directors[:3]) if directors else None,
                "top_cast": "|".join(p.get("name", "") for p in cast[:3]) if cast else None,
                "production_companies": "|".join(c.get("name", "") for c in data.get("production_companies", [])) or None,
                "production_countries": "|".join(c.get("name", "") for c in data.get("production_countries", [])) or None,
                "keywords": "|".join(k.get("name", "") for k in data.get("keywords", {}).get("keywords", [])) or None,
            }
        except Exception:
            await asyncio.sleep(1 + attempt * 2)
    return None


async def run_scrape(pending_rows):
    semaphore = asyncio.Semaphore(SEMAPHORE_LIMIT)
    buffer = []
    success = 0
    failed = 0

    async with aiohttp.ClientSession(headers=HEADERS) as session:
        tasks = [fetch_tmdb_async(session, row, semaphore) for row in pending_rows]

        for i, coro in enumerate(asyncio.as_completed(tasks), 1):
            result = await coro
            if result:
                buffer.append(result)
                success += 1
            else:
                failed += 1

            if len(buffer) >= SAVE_EVERY:
                append_checkpoint(CHECKPOINT_FILE, buffer)
                print(f"Checkpoint | done={i:,} success={success:,} failed={failed:,}")
                buffer = []

            if i % 2000 == 0:
                print(f"Progress {i:,}/{len(pending_rows):,}")

    append_checkpoint(CHECKPOINT_FILE, buffer)
    print(f"Done. Success: {success:,} | Failed: {failed:,}")

# Run it
rows = pending_df.to_dict("records")
await run_scrape(rows)

Checkpoint | done=254 success=250 failed=4
Checkpoint | done=508 success=500 failed=8
Checkpoint | done=761 success=750 failed=11
Checkpoint | done=1,015 success=1,000 failed=15
Checkpoint | done=1,267 success=1,250 failed=17
Checkpoint | done=1,521 success=1,500 failed=21
Checkpoint | done=1,775 success=1,750 failed=25
Progress 2,000/81,907
Checkpoint | done=2,027 success=2,000 failed=27
Checkpoint | done=2,278 success=2,250 failed=28
Checkpoint | done=2,530 success=2,500 failed=30
Checkpoint | done=2,787 success=2,750 failed=37
Checkpoint | done=3,041 success=3,000 failed=41
Checkpoint | done=3,296 success=3,250 failed=46
Checkpoint | done=3,551 success=3,500 failed=51
Checkpoint | done=3,802 success=3,750 failed=52
Progress 4,000/81,907
Checkpoint | done=4,054 success=4,000 failed=54
Checkpoint | done=4,306 success=4,250 failed=56
Checkpoint | done=4,557 success=4,500 failed=57
Checkpoint | done=4,814 success=4,750 failed=64
Checkpoint | done=5,070 success=5,000 failed=70
Checkpoint

In [ ]:
'''
# Run TMDB requests in parallel.
# This loop continues through the whole selected subset and saves progress every SAVE_EVERY successful rows.
rows = pending_df.to_dict("records")
buffer = []
success_count = 0
failed_count = 0
start_time = time.time()

print(f"Starting TMDB enrichment for {len(rows):,} pending movies...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_tmdb_metadata, row) for row in rows]

    for completed, future in enumerate(as_completed(futures), start=1):
        result = future.result()

        if result is None:
            failed_count += 1
        else:
            buffer.append(result)
            success_count += 1

        # Checkpoint saves partial progress before the full scrape completes.
        if len(buffer) >= SAVE_EVERY:
            append_checkpoint(CHECKPOINT_FILE, buffer)
            buffer = []
            elapsed_min = (time.time() - start_time) / 60
            print(f"Saved checkpoint | completed={completed:,} success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

        # Progress display for long runs.
        if completed % 500 == 0 or completed == len(rows):
            elapsed_min = (time.time() - start_time) / 60
            print(f"Progress {completed:,}/{len(rows):,} | success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

# Final checkpoint save for the remaining rows.
append_checkpoint(CHECKPOINT_FILE, buffer)

print("TMDB enrichment finished.")
print("Successful new rows:", success_count)
print("Failed or skipped rows:", failed_count)
'''

Starting TMDB enrichment for 86,411 pending movies...
Saved checkpoint | completed=250 success=250 failed=0 elapsed=0.1 min
Saved checkpoint | completed=500 success=500 failed=0 elapsed=0.2 min
Progress 500/86,411 | success=500 failed=0 elapsed=0.2 min
Saved checkpoint | completed=751 success=750 failed=1 elapsed=0.3 min
Progress 1,000/86,411 | success=999 failed=1 elapsed=0.4 min
Saved checkpoint | completed=1,001 success=1,000 failed=1 elapsed=0.4 min
Saved checkpoint | completed=1,251 success=1,250 failed=1 elapsed=0.6 min
Progress 1,500/86,411 | success=1,499 failed=1 elapsed=0.7 min
Saved checkpoint | completed=1,501 success=1,500 failed=1 elapsed=0.7 min
Saved checkpoint | completed=1,751 success=1,750 failed=1 elapsed=0.8 min
Progress 2,000/86,411 | success=1,999 failed=1 elapsed=0.9 min
Saved checkpoint | completed=2,001 success=2,000 failed=1 elapsed=0.9 min
Saved checkpoint | completed=2,251 success=2,250 failed=1 elapsed=1.0 min
Progress 2,500/86,411 | success=2,499 failed=1

: 

: 

In [ ]:
# Load the full checkpoint after scraping.
# Duplicate tmdb_id rows are removed in case a run was restarted near a checkpoint.
tmdb_metadata_df = pd.read_csv(CHECKPOINT_FILE)
tmdb_metadata_df = tmdb_metadata_df.drop_duplicates(subset=["tmdb_id"], keep="last")

# Re-save the cleaned checkpoint.
tmdb_metadata_df.to_csv(CHECKPOINT_FILE, index=False)

print("Total saved TMDB rows:", len(tmdb_metadata_df))
tmdb_metadata_df.head()

Total saved TMDB rows: 85204


,movieId,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
0,1,862,30000000.0,401157969.0,81.0,1995-11-22,en,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,Pixar,United States of America,rescue|friendship|mission|jealousy|villain|bul...
1,30,37557,2300000.0,NaN,108.0,1995-09-14,zh,Zhang Yimou,Gong Li|Li Baotian|Sun Chun,Alpha Films|La Sept Cinéma|UGC|Shanghai Film S...,China|France,"shanghai, china|chinese mafia|coming of age|mi..."
2,11,9087,62000000.0,107879496.0,113.0,1995-11-17,en,Rob Reiner,Michael Douglas|Annette Bening|Martin Sheen,Castle Rock Entertainment|Universal Pictures|W...,United States of America,new love|usa president|the white house|holiday...
3,10,710,60000000.0,352194034.0,130.0,1995-11-16,en,Martin Campbell,Pierce Brosnan|Sean Bean|Izabella Scorupco,EON Productions,United Kingdom,computer virus|cuba|falsely accused|secret int...
4,3,15602,25000000.0,71518503.0,101.0,1995-12-22,en,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,Lancaster Gate|Warner Bros. Pictures,United States of America,fishing|sequel|old man|best friend|wedding|ita...


In [ ]:
# Merge TMDB metadata back into movies_df using a left join on movieId.
# This guarantees ALL movies from movies.csv appear in the output.
# Movies without a tmdbId (e.g. movieId=126) will have NaN in enrichment columns.
movies_enriched_df = movies_df.merge(
    tmdb_metadata_df[["movieId"] + [c for c in FIELDNAMES if c != "movieId"]],
    on="movieId",
    how="left"
)

movies_enriched_df.to_csv(ENRICHED_MOVIES_FILE, index=False)

print("Saved enriched movies file:", ENRICHED_MOVIES_FILE)
print("Total rows (should equal movies.csv):", len(movies_enriched_df))
print("Movies WITHOUT tmdb data (no tmdbId or 404):",
      movies_enriched_df["tmdb_id"].isna().sum())
movies_enriched_df.head()


Saved enriched movies file: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\movies_enriched_selected.csv
Total rows (should equal movies.csv): 86537
Movies WITHOUT tmdb data (no tmdbId or 404): 1333


,movieId,title,genres,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,30000000.0,401157969.0,81.0,1995-11-22,en,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,Pixar,United States of America,rescue|friendship|mission|jealousy|villain|bul...
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,65000000.0,262821940.0,104.0,1995-12-15,en,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce,TriStar Pictures|Interscope Communications|Tei...,United States of America,giant insect|board game|disappearance|jungle|r...
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,25000000.0,71518503.0,101.0,1995-12-22,en,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,Lancaster Gate|Warner Bros. Pictures,United States of America,fishing|sequel|old man|best friend|wedding|ita...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,16000000.0,81452156.0,127.0,1995-12-22,en,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,20th Century Fox,United States of America,based on novel or book|single mother|divorce|a...
4,5,Father of the Bride Part II (1995),Comedy,11862.0,NaN,76594107.0,106.0,1995-12-08,en,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,Touchstone Pictures|Sandollar Productions,United States of America,daughter|baby|parent child relationship|midlif...


In [ ]:
# Data quality summary for the enriched metadata columns.
# number of missing values each variable contains
enrichment_cols = [c for c in FIELDNAMES if c in movies_enriched_df.columns]
print(movies_enriched_df[enrichment_cols].isna().sum().sort_values(ascending=False))


revenue                 70705
budget                  70470
keywords                24932
production_companies    11512
production_countries     5573
top_cast                 4464
runtime                  2018
director                 1834
release_date             1367
tmdb_id                  1333
original_language        1333
movieId                     0
dtype: int64


In [ ]:
# checking the rows without the tmdbID
movies_enriched_df[movies_enriched_df["tmdb_id"].isna()]

,movieId,title,genres,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
596,604,Criminals (1996),Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
706,721,Halfmoon (Paul Bowles - Halbmond) (1995),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
715,730,Low Life (1994),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
754,770,Costa Brava (1946),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
775,791,"Last Klezmer: Leopold Kozlowski, His Life and ...",Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85888,286851,Werewolf Women of the S.S. (2007),Comedy|Horror,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86093,287439,Muppet*Vision 3-D (1991),(no genres listed),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86282,288069,American Masters: Miles Davis - Birth of the Cool,Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86365,288447,A House A Little Too Quiet,(no genres listed),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# checking how many rows (not counting the ones without tmdbID) have at least one missing value in the four columns:
check_cols = ["top_cast", "runtime", "director", "release_date"]

has_tmdb = movies_enriched_df[movies_enriched_df["tmdb_id"].notna()]
has_any_missing = has_tmdb[check_cols].isna().any(axis=1).sum()

print(f"Movies with valid tmdb_id: {len(has_tmdb):,}")
print(f"  All 4 fields complete: {len(has_tmdb) - has_any_missing:,}")
print(f"  At least one missing: {has_any_missing:,}")
print()
print(has_tmdb[check_cols].isna().sum())

Movies with valid tmdb_id: 85,204
  All 4 fields complete: 81,175
  At least one missing: 4,029

top_cast        3131
runtime          685
director         501
release_date      34
dtype: int64
